# monomap v2 on Colab (GPU)

Runs the full metric / CAD / nav pipeline on a free Colab GPU, so your laptop's Smart App Control stays ON.

**You provide:** (1) `monomap_src.zip` (the `src/` folder, from `monomap/colab/`), and (2) a video clip **or** a PoseCam export folder.

**You get back:** `mesh.obj/stl/ply`, `map_metric.ply/.pcd` + `poses.tum`, `planes.json`/`boxes.json`, and `floormap.pgm/.yaml` + `occupancy_3d.npy`.

Run the cells top to bottom. First set **Runtime → Change runtime type → GPU (T4)**.

## 1. Install deps (torch is pre-installed on Colab with CUDA)

In [ ]:
!apt-get -qq install -y libgl1 libgomp1 >/dev/null
!pip -q install open3d opencv-python-headless pillow 'transformers>=4.45' huggingface_hub
import torch; print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
import open3d as o3d; print('open3d', o3d.__version__)

## 2. Upload the code (`monomap_src.zip`)

In [ ]:
import zipfile, os
from google.colab import files
up = files.upload()           # pick monomap_src.zip
zf = [k for k in up if k.endswith('.zip')][0]
os.makedirs('/content/monomap', exist_ok=True)
with zipfile.ZipFile(zf) as z: z.extractall('/content/monomap')
print('extracted ->', os.listdir('/content/monomap/src')[:12])

## 3. Get your input
Pick ONE. Big videos: use Google Drive (put the clip in `MyDrive`). Small clips: direct upload.

In [ ]:
# --- Option A: Google Drive (recommended for larger files) ---
from google.colab import drive; drive.mount('/content/drive')
VIDEO = '/content/drive/MyDrive/clip1.mp4'   # <-- edit to your file
POSECAM = None                               # or '/content/drive/MyDrive/posecam_scan1'

# --- Option B: direct upload a small clip (uncomment) ---
# from google.colab import files; u = files.upload(); VIDEO = '/content/' + list(u)[0]; POSECAM = None
print('VIDEO =', VIDEO, '| POSECAM =', POSECAM)

## 4. Run the pipeline
`--video` = plain clip (relative scale). `--posecam` = ARCore export (metric). Edit `NAME`/`--stride` as needed.

In [ ]:
NAME = 'scan1'
import os; os.chdir('/content/monomap')
if POSECAM:
    !python src/run_pipeline.py --posecam "{POSECAM}" --name {NAME} --stride 2 --target-tris 400000
else:
    !python src/run_pipeline.py --video "{VIDEO}" --name {NAME} --stride 2 --target-tris 400000

## 5. Download the results

In [ ]:
import shutil
out = f'/content/monomap/output/{NAME}_v2'
print('contents:', os.listdir(out))
shutil.make_archive(f'/content/{NAME}_v2', 'zip', out)
from google.colab import files; files.download(f'/content/{NAME}_v2.zip')

## What's in the zip
| file | what | open with |
|---|---|---|
| `mesh.obj` / `mesh.stl` / `mesh.ply` | watertight surface | Blender / MeshLab / any slicer |
| `map_metric.ply` / `.pcd` + `poses.tum` | cloud + trajectory | CloudCompare / ROS tooling |
| `planes.json` / `boxes.json` / `primitives.ply` | floor/walls + object boxes | your code / viewer |
| `floormap.png/.pgm/.yaml`, `occupancy_3d.npy` | 2D costmap + 3D voxels | ROS map_server / planner |

**ARCore (metric) path:** record with the PoseCam app, copy its export folder (the `RGB_*.mp4`, `AR_Pose_*.txt`, `intrinsics.json`) to Drive, set `POSECAM` to that folder in cell 3, and re-run.

**Quality upgrade:** on this bigger GPU you can later swap the depth stage to *Video Depth Anything* (temporally consistent) — ask and I'll wire it into `depth.py`.